In [3]:
# ============================================================================
# CELL 2: LOAD VARIABLES (ضعها في بداية Notebook الجديد)
# ============================================================================

import pickle
from pathlib import Path
from datetime import datetime

import pandas as pd

print("=" * 80)
print("📥 LOADING SAVED VARIABLES")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# 📂 STEP 1: Find save folder
# ═══════════════════════════════════════════════════════════════════════════

save_folder = Path.home() / "Desktop" / "notebook_variables"

if not save_folder.exists():
    print()
    print("❌ No saved sessions found!")
    print(f"   Expected location: {save_folder}")
    print()
    print("💡 First run CELL 1 in your source notebook to save variables")
    print("=" * 80)
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # 📋 STEP 2: List available sessions
    # ═══════════════════════════════════════════════════════════════════════════

    sessions = sorted([d for d in save_folder.iterdir() if d.is_dir()], reverse=True)

    if len(sessions) == 0:
        print()
        print("❌ No sessions found in folder!")
        print(f"   Folder exists but is empty: {save_folder}")
        print()
        print("💡 Run CELL 1 in your source notebook to create a session")
        print("=" * 80)
    else:
        print()
        print(f"📂 Found {len(sessions)} saved session(s):")
        print()
        print("-" * 80)

        # Display available sessions
        for idx, session in enumerate(sessions, 1):
            metadata_path = session / "metadata.pkl"

            if metadata_path.exists():
                try:
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)

                    dt = metadata.get('datetime', metadata.get('timestamp', 'Unknown'))
                    var_count = metadata.get('success_count',
                               len([v for v in metadata.get('saved_variables', {}).values()
                                   if v.get('saved', False)]))

                    print(f"  {idx}. {session.name}")
                    print(f"     Date: {dt}")
                    print(f"     Variables: {var_count}")

                    # Show variable names
                    vars_list = [k for k, v in metadata.get('saved_variables', {}).items()
                                if v.get('saved', False)]
                    if len(vars_list) > 0:
                        print(f"     Contains: {', '.join(vars_list[:5])}")
                        if len(vars_list) > 5:
                            print(f"               ... and {len(vars_list)-5} more")
                    print()
                except:
                    print(f"  {idx}. {session.name} (metadata error)")
                    print()
            else:
                print(f"  {idx}. {session.name} (no metadata)")
                print()

        print("-" * 80)

        # ═══════════════════════════════════════════════════════════════════════════
        # 🎯 STEP 3: Choose session to load
        # ═══════════════════════════════════════════════════════════════════════════

        choice = input("\nEnter session number to load (press Enter for latest): ").strip()

        if choice == '':
            choice = '1'

        try:
            session_idx = int(choice) - 1

            if session_idx < 0 or session_idx >= len(sessions):
                print(f"\n❌ Invalid choice! Must be 1-{len(sessions)}")
                print("=" * 80)
            else:
                selected_session = sessions[session_idx]

                print()
                print("=" * 80)
                print(f"📥 Loading session: {selected_session.name}")
                print("=" * 80)
                print()

                # ═══════════════════════════════════════════════════════════════════════════
                # 📦 STEP 4: Load metadata
                # ═══════════════════════════════════════════════════════════════════════════

                metadata_path = selected_session / "metadata.pkl"

                if metadata_path.exists():
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)
                else:
                    print("⚠️  No metadata found - will try to load all .pkl files")
                    metadata = {'saved_variables': {}}

                # ═══════════════════════════════════════════════════════════════════════════
                # 💾 STEP 5: Load variables
                # ═══════════════════════════════════════════════════════════════════════════

                loaded_count = 0
                failed_count = 0

                saved_vars = metadata.get('saved_variables', {})

                if len(saved_vars) == 0:
                    # No metadata, try all .pkl files
                    pkl_files = list(selected_session.glob("*.pkl"))
                    print(f"Found {len(pkl_files)} .pkl files (excluding metadata)")
                    print()

                    for pkl_file in pkl_files:
                        if pkl_file.name != "metadata.pkl":
                            var_name = pkl_file.stem  # filename without .pkl

                            try:
                                with open(pkl_file, 'rb') as f:
                                    var_value = pickle.load(f)

                                globals()[var_name] = var_value

                                size_kb = pkl_file.stat().st_size / 1024
                                size_str = f"{size_kb:.1f} KB" if size_kb < 1024 else f"{size_kb/1024:.1f} MB"
                                var_type = type(var_value).__name__

                                print(f"✅ {var_name:<25} | {var_type:<15} | {size_str}")
                                loaded_count += 1

                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1
                else:
                    # Use metadata
                    for var_name, var_info in saved_vars.items():
                        if var_info.get('saved', False):
                            try:
                                file_path = selected_session / f"{var_name}.pkl"

                                with open(file_path, 'rb') as f:
                                    var_value = pickle.load(f)

                                # Load into global scope
                                globals()[var_name] = var_value

                                print(f"✅ {var_name:<25} | {var_info.get('type', 'Unknown'):<15} | {var_info.get('size', 'Unknown')}")
                                loaded_count += 1

                            except FileNotFoundError:
                                print(f"❌ {var_name:<25} | FILE NOT FOUND")
                                failed_count += 1
                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1

                # ═══════════════════════════════════════════════════════════════════════════
                # ✅ STEP 6: Summary
                # ═══════════════════════════════════════════════════════════════════════════

                print()
                print("=" * 80)
                print("✅ LOAD COMPLETE!")
                print("=" * 80)
                print(f"✅ Loaded:  {loaded_count} variables")
                print(f"❌ Failed:  {failed_count} variables")
                print("=" * 80)
                print()
                print("💡 Variables are now available in this notebook!")
                print("   Example: print(combined_df.head())")
                print("=" * 80)

        except ValueError:
            print()
            print("❌ Invalid input! Please enter a number")
            print("=" * 80)

📥 LOADING SAVED VARIABLES

📂 Found 29 saved session(s):

--------------------------------------------------------------------------------
  1. session_20260119_145703
     Date: 2026-01-19 14:57:45
     Variables: 7
     Contains: combined_df, combined_df_updated, df, inventory_df, inventory_df_tagropa
               ... and 2 more

  2. session_20260119_145440
     Date: 2026-01-19 14:55:29
     Variables: 5
     Contains: combined_df, combined_df_updated, df, inventory_df, inventory_df_tagropa

  3. session_20260119_123201
     Date: 2026-01-19 12:32:51
     Variables: 6
     Contains: combined_df, combined_df_updated, df, inventory_df, sku_df
               ... and 1 more

  4. session_20260119_122934
     Date: 2026-01-19 12:30:21
     Variables: 4
     Contains: combined_df, combined_df_updated, df, inventory_df

  5. session_20260119_122517
     Date: 2026-01-19 12:26:21
     Variables: 4
     Contains: combined_df, combined_df_updated, df, inventory_df

  6. session_20260119_121

In [16]:
combined_df.head()


,Section,Section Name,bal Value,bal Qty,U Price,Client,Client Name,Outlet,Outlet Name,Date,...,Nb. Days (Avail. Balance),SKU-STATUES-Mahmoud,SKU STATUES -Python,OLD_QTY,First_Inv_Year,Unit_Price,Total_Cost,Total_Profit,Profit_Margin_%,Sales_Type
0,3813,جاكار امبيوريو,513.510010,51.299999,10.01,4.600000e+09,عائله,46.0,صالة اليمن للتخفيضات,2019-01-01,...,120.0,clearance,Loss,1040.890015,2013.0,10.009942,923.399963,-409.889954,-79.821220,FAMILY
1,N116,,443.739990,32.000000,13.87,4.400002e+09,زكي,44.0,النجمة جدة,2019-01-01,...,NaN,None,None,NaN,NaN,13.866875,NaN,NaN,NaN,FAMILY
2,3884,أورقانزا تايتان,316.799988,12.000000,26.40,2.300000e+09,عـــائـــلــة,23.0,صالة العليا,2019-01-01,...,120.0,clearance,Sporadic (Inactive),814.539978,2013.0,26.400000,180.000000,136.799988,43.181816,FAMILY
3,4883,قطن استر بلس,112.059998,4.500000,24.90,4.000000e+08,مؤسسة لؤي نايل بوقري للتشغيل والصيانة,4.0,مركز مكة,2019-01-01,...,120.0,activation,Dead (Stock Available),210.350006,2018.0,24.902222,54.000000,58.059998,51.811527,FAMILY
4,4410,دانتيل بامبو,-133.488007,-3.000000,44.50,7.600000e+09,شادن ستائر ومفروشات,76.0,مركز قطر,2019-01-01,...,120.0,activation,Stock-Out (Was Profitable),2390.750000,2016.0,44.496002,-24.000000,-109.488007,82.020859,FAMILY


In [25]:
# # supplier_df=combined_df[['CATEGORY'  , 'Section' , 'SKU' , 'CURRENT_STOCK' , 'bal Qty' , 'bal Value' , 'Date' , 'Total_Profit' , 'Total_Cost' , 'Supplier' , 'Cost' , 'Sales_Type']].copy()
# #
# #
# # supplier_df['Year']=pd.to_datetime(supplier_df['Date']).dt.year
# # supplier_df['Total STK Cost']=supplier_df['CURRENT_STOCK'] * supplier_df['Cost']
# #
# # supplier_df = supplier_df[supplier_df['Supplier'] == "D'DECOR HOME FABRICS PVT.LTD"]
# #
# #
# # final_df_all=supplier_df.groupby(
# #     ['SKU' , 'Section' , 'Year' , 'Sales_Type'] , as_index=False
# # ).agg(
# #     total_stock=('CURRENT_STOCK', 'first'),
# #     total_sales_value=('bal Value', 'sum'),
# #     total_sales_qty=('bal Qty', 'sum'),
# #     stock_value=('Total STK Cost', 'first'),
# #     total_cost=('Total_Cost', 'sum')
# # )
# # final_df_all['Gross margine%'] =((final_df_all['total_sales_value'] - final_df_all['total_cost'] )/final_df_all['total_cost']  )*100
# #
# # final_df_all.head()
# #
# # final_df_all.to_excel('Sales - supplier.xlsx')
#
#
#
#
#
#
# # عمل نسخة من الأعمدة المطلوبة
# supplier_df = combined_df[['CATEGORY', 'Section', 'SKU', 'CURRENT_STOCK',
#                            'bal Qty', 'bal Value', 'Date', 'Total_Profit',
#                            'Total_Cost', 'Supplier', 'Cost', 'Sales_Type']].copy()
#
# # إضافة السنة وحساب تكلفة المخزون
# supplier_df['Year'] = pd.to_datetime(supplier_df['Date']).dt.year
# supplier_df['Total_STK_Cost'] = supplier_df['CURRENT_STOCK'] * supplier_df['Cost']
#
# # فلترة المورد المطلوب
# supplier_df = supplier_df[supplier_df['Supplier'] == "D'DECOR HOME FABRICS PVT.LTD"]
#
# # تحديد الأعمدة الرقمية للـ aggregation
# numeric_cols = ['CURRENT_STOCK', 'bal Value', 'bal Qty', 'Total_STK_Cost', 'Total_Cost', 'Total_Profit', 'Cost']
#
# # groupby مع aggregate للأرقام و first لباقي الأعمدة
# final_df_all = supplier_df.groupby(['SKU', 'Section', 'Year', 'Sales_Type'], as_index=False).agg(
#     total_stock=('CURRENT_STOCK', 'sum'),
#     total_sales_value=('bal Value', 'sum'),
#     total_sales_qty=('bal Qty', 'sum'),
#     stock_value=('Total_STK_Cost', 'sum'),
#     total_cost=('Total_Cost', 'sum'),
#     # الأعمدة الغير رقمية: ناخد أول قيمة
#     CATEGORY=('CATEGORY', 'first'),
#     Supplier=('Supplier', 'first'),
#     Date=('Date', 'first')
# )
#
# # حساب الهامش
# final_df_all['Gross_margin_%'] = ((final_df_all['total_sales_value'] - final_df_all['total_cost']) / final_df_all['total_sales_value']) * 100
#
# # حفظ الملف
# final_df_all.to_excel('Sales - supplier -21.xlsx', index=False)




supplier_df = combined_df[['CATEGORY', 'Section', 'SKU', 'CURRENT_STOCK',
                          'bal Qty', 'bal Value', 'Date', 'Total_Profit',
                          'Total_Cost', 'Supplier', 'Cost', 'Sales_Type']].copy()

supplier_df['Year'] = pd.to_datetime(supplier_df['Date']).dt.year
supplier_df['Total_STK_Cost'] = supplier_df['CURRENT_STOCK'] * supplier_df['Cost']

supplier_df = supplier_df[supplier_df['Supplier'] == "D'DECOR HOME FABRICS PVT.LTD"]

final_df_all = supplier_df.groupby(['SKU', 'Section', 'Year', 'Sales_Type'], as_index=False).agg(
    total_sales_value=('bal Value', 'sum'),
    total_sales_qty=('bal Qty', 'sum'),
    total_cost=('Total_Cost', 'sum'),
    CATEGORY=('CATEGORY', 'first'),
    Supplier=('Supplier', 'first'),
    Cost=('Cost', 'first')
)

# نجيب المخزون مرة واحدة لكل SKU (غير مرتبط بالسنة ولا sales_type)
stock_df = supplier_df.groupby(['SKU', 'Section']).agg(
    current_stock=('CURRENT_STOCK', 'last')
).reset_index()

# ندمج مخزون الاستوك مع الداتا الرئيسية
final_df_all = final_df_all.merge(stock_df, on=['SKU', 'Section'], how='left')

# حساب stock_value = current_stock * cost
final_df_all['stock_value'] = final_df_all['current_stock'] * final_df_all['Cost']

# حساب الهامش كالنسبة
final_df_all['Gross_margin_%'] = ((final_df_all['total_sales_value'] - final_df_all['total_cost']) / final_df_all['total_sales_value']) * 100

final_df_all.to_excel('Sales - supplier_fixed_stock.xlsx', index=False)



In [17]:
import pandas as pd



supplier_df=combined_df[['CATEGORY'  , 'Section' , 'SKU' , 'CURRENT_STOCK' , 'bal Qty' , 'bal Value' , 'Date' , 'Total_Profit' , 'Total_Cost' , 'Supplier' , 'Cost' , 'Sales_Type']].copy()


supplier_df['Year']=pd.to_datetime(supplier_df['Date']).dt.year
supplier_df['Total STK Cost']=supplier_df['CURRENT_STOCK'] * supplier_df['Cost']

supplier_df = supplier_df[supplier_df['Supplier'] == "D'DECOR HOME FABRICS PVT.LTD"]


final_df_all=supplier_df.groupby(
    ['SKU' , 'Section' , 'Year' , 'Sales_Type'] , as_index=False
).agg(
    total_stock=('CURRENT_STOCK', 'first'),
    total_sales_value=('bal Value', 'sum'),
    total_sales_qty=('bal Qty', 'sum'),
    stock_value=('Total STK Cost', 'first'),
    total_cost=('Total_Cost', 'sum')
)
final_df_all['Gross margine%'] =((final_df_all['total_sales_value'] - final_df_all['total_cost'] )/final_df_all['total_cost']  )*100

final_df_all.head()


,SKU,Section,Year,Sales_Type,total_stock,total_sales_value,total_sales_qty,stock_value,total_cost,Gross margine%
0,2308010012,2308,2022.0,CONTRACT,0.0,860.00,230.0,0.0,5520.0,-84.42029
1,2308010013,2308,2022.0,FAMILY,5.7,180.00,48.0,136.8,1152.0,-84.37500
2,2308010016,2308,2025.0,FAMILY,295.6,0.00,0.2,7094.4,4.8,-100.00000
3,2308010019,2308,2025.0,FAMILY,283.0,0.00,0.2,6792.0,4.8,-100.00000
4,2308010022,2308,2022.0,FAMILY,159.8,148.05,4.7,3835.2,112.8,31.25000


In [16]:
import pandas as pd




# ====================================================================
# SKU NORMALIZATION FUNCTION (IMPORTANT)
# ====================================================================

def normalize_sku(series):
    return (
        series
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)   # remove trailing .0
        .str.replace(r'\.0+', '', regex=True)
        .str.replace(r'\s+', '', regex=True)   # remove any spaces inside
        .str.strip()
        .str.upper()
    )



# =========================
# 1️⃣ Prepare Data
# =========================
supplier_df = combined_df[
    ['SKU','Section','CATEGORY','CURRENT_STOCK',
     'bal Qty','bal Value','Date','Total_Cost',
     'Supplier','Cost','Sales_Type']
]

supplier_df = supplier_df[supplier_df['Supplier'] == "D'DECOR HOME FABRICS PVT.LTD"]

supplier_df['Year'] = pd.to_datetime(supplier_df['Date']).dt.year
supplier_df['Stock_Cost'] = supplier_df['CURRENT_STOCK'] * supplier_df['Cost']

# =========================
# 2️⃣ Pivot Sales Qty (Year + Sales Type)
# =========================
sales_qty_pivot = pd.pivot_table(
    supplier_df,
    values='bal Qty',
    index=['SKU','Section','CATEGORY'],
    columns=['Year','Sales_Type'],
    aggfunc='sum',
    fill_value=0
)

sales_qty_pivot.columns = [
    f"Sales_{year}_{stype}_Qty"
    for year, stype in sales_qty_pivot.columns
]

sales_qty_pivot = sales_qty_pivot.reset_index()

# =========================
# 3️⃣ Current Stock
# =========================
stock_df = (
    supplier_df
    .groupby(['SKU','Section','CATEGORY'], as_index=False)
    .agg(
        stk=('CURRENT_STOCK','first'),
        stock_cost=('Stock_Cost','first')
    )
)

# =========================
# 4️⃣ Sales Value + Margin (2025)
# =========================
sales_2025 = supplier_df[supplier_df['Year'] == 2025]

sales_2025_df = (
    sales_2025
    .groupby(['SKU','Section','CATEGORY'], as_index=False)
    .agg(
        sales_value_2025=('bal Value','sum'),
        total_cost_2025=('Total_Cost','sum')
    )
)

sales_2025_df['Profit_Margin_%'] = (
    (sales_2025_df['sales_value_2025'] - sales_2025_df['total_cost_2025'])
    / sales_2025_df['total_cost_2025']
) * 100

# =========================
# 5️⃣ Final Merge
# =========================
final_df = (
    stock_df
    .merge(sales_qty_pivot, on=['SKU','Section','CATEGORY'], how='left')
    .merge(sales_2025_df, on=['SKU','Section','CATEGORY'], how='left')
)

# =========================
# 6️⃣ Export to Excel
# =========================
file_name = 'Mahmoud_Sales_Report.xlsx'

with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
    final_df.to_excel(writer, sheet_name="D'DECOR HOME FABRICS PVT.LTD", index=False)

print(f"Excel file created: {file_name}")


Excel file created: Mahmoud_Sales_Report.xlsx


In [17]:
# ════════════════════════════════════════════════════════════════════════════
# PROFESSIONAL SUPPLIER ANALYSIS SYSTEM - ENHANCED
# Vectorized + Realistic Metrics + PDF Export + Dashboard
# ════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PDF Export
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import (SimpleDocTemplate, Table, TableStyle, Paragraph,
                                    Spacer, PageBreak, Image as RLImage)
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib import colors as rl_colors
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
    from reportlab.pdfgen import canvas
    PDF_OK = True
except:
    PDF_OK = False
    print("⚠️  PDF export not available - install: pip install reportlab")

print("=" * 80)
print("📊 PROFESSIONAL SUPPLIER ANALYSIS SYSTEM")
print("=" * 80)


# ════════════════════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def normalize_sku(series):
    """Normalize SKU format for consistent matching"""
    return (
        series
        .astype(str)
        .str.replace(r'\.0+$', '', regex=True)
        .str.replace(r'\s+', '', regex=True)
        .str.strip()
        .str.upper()
    )


# ════════════════════════════════════════════════════════════════════════════
# MAIN SUPPLIER ANALYSIS ENGINE (VECTORIZED)
# ════════════════════════════════════════════════════════════════════════════

class SupplierAnalyzer:
    """Professional supplier analysis with vectorized calculations"""

    def __init__(self, combined_df, forecast_df=None, stock_df=None):
        self.df = combined_df.copy()
        self.forecast_df = forecast_df
        self.stock_df = stock_df

        # Ensure date column
        self.df['Date'] = pd.to_datetime(self.df['Date'])

        # Parse Collection if not present
        if 'Collection' not in self.df.columns:
            self.df['Collection'] = self.df['SKU'].astype(str).str[:4]

        # Output folder
        self.output_folder = Path.home() / "Desktop" / "supplier_analysis"
        self.output_folder.mkdir(exist_ok=True)

    def analyze_supplier(self, supplier_name, adjusted_cost_file=None):
        """
        Comprehensive supplier analysis

        Parameters:
        -----------
        supplier_name : str
            Supplier name to analyze
        adjusted_cost_file : str, optional
            Path to adjusted cost Excel file

        Returns:
        --------
        result_df : DataFrame
            Complete supplier analytics by SKU
        """

        print("\n" + "=" * 80)
        print(f"📊 ANALYZING SUPPLIER: {supplier_name}")
        print("=" * 80)

        # ═══════════════════════════════════════════════════════════════
        # 1. Filter Supplier Data
        # ═══════════════════════════════════════════════════════════════

        print("\n🔍 Step 1: Filtering supplier data...")

        supplier_df = self.df[self.df['Supplier'] == supplier_name].copy()

        if len(supplier_df) == 0:
            print(f"❌ No data found for supplier: {supplier_name}")
            return None

        print(f"   ✅ Found {len(supplier_df):,} records")
        print(f"   ✅ SKUs: {supplier_df['SKU'].nunique():,}")
        print(f"   ✅ Period: {supplier_df['Date'].min().date()} → {supplier_df['Date'].max().date()}")

        # ═══════════════════════════════════════════════════════════════
        # 2. Load Adjusted Cost (if provided)
        # ═══════════════════════════════════════════════════════════════

        if adjusted_cost_file and Path(adjusted_cost_file).exists():
            print("\n💰 Step 2: Loading adjusted costs...")

            try:
                adj_df = pd.read_excel(adjusted_cost_file)

                # Auto-detect columns
                sku_col = next((c for c in adj_df.columns if 'sku' in c.lower()), None)
                cost_col = next((c for c in adj_df.columns
                               if 'cost' in c.lower() or 'سعر' in c.lower()), None)

                if sku_col and cost_col:
                    adj_df = adj_df[[sku_col, cost_col]].copy()
                    adj_df.columns = ['SKU', 'Adjusted_Cost']

                    # Normalize SKUs
                    adj_df['SKU'] = normalize_sku(adj_df['SKU'])
                    supplier_df['SKU'] = normalize_sku(supplier_df['SKU'])

                    adj_df['Adjusted_Cost'] = pd.to_numeric(adj_df['Adjusted_Cost'], errors='coerce')

                    # Merge
                    supplier_df = supplier_df.merge(
                        adj_df[['SKU', 'Adjusted_Cost']],
                        on='SKU',
                        how='left'
                    )

                    # Use adjusted cost where available
                    supplier_df['Final_Cost'] = supplier_df['Adjusted_Cost'].fillna(supplier_df['Cost'])

                    adj_count = supplier_df['Adjusted_Cost'].notna().sum()
                    print(f"   ✅ Applied adjusted costs to {adj_count:,} SKUs")
                else:
                    print("   ⚠️  Could not detect SKU/Cost columns")
                    supplier_df['Final_Cost'] = supplier_df['Cost']

            except Exception as e:
                print(f"   ⚠️  Error loading adjusted cost: {e}")
                supplier_df['Final_Cost'] = supplier_df['Cost']
        else:
            supplier_df['Final_Cost'] = supplier_df['Cost']

        # ═══════════════════════════════════════════════════════════════
        # 3. Recalculate Financials (Vectorized)
        # ═══════════════════════════════════════════════════════════════

        print("\n💵 Step 3: Calculating financials...")

        supplier_df['Total_Cost'] = supplier_df['Final_Cost'] * supplier_df['bal Qty']
        supplier_df['Total_Profit'] = supplier_df['bal Value'] - supplier_df['Total_Cost']
        supplier_df['Margin_%'] = np.where(
            supplier_df['bal Value'] > 0,
            (supplier_df['Total_Profit'] / supplier_df['bal Value']) * 100,
            0
        )

        # Add time dimensions
        supplier_df['Year'] = supplier_df['Date'].dt.year
        supplier_df['Month'] = supplier_df['Date'].dt.month
        supplier_df['Quarter'] = supplier_df['Date'].dt.quarter

        print("   ✅ Financials calculated")

        # ═══════════════════════════════════════════════════════════════
        # 4. Build SKU-Level Analytics (Vectorized)
        # ═══════════════════════════════════════════════════════════════

        print("\n📦 Step 4: Building SKU-level analytics...")

        # Get latest record per SKU for static attributes
        latest = supplier_df.sort_values('Date').groupby('SKU').last().reset_index()

        # Keep only needed columns
        base_cols = ['SKU', 'Section', 'CATEGORY', 'Supplier', 'Collection', 'Final_Cost']
        stock_cols = ['CURRENT_STOCK', 'OUTSTANDING', 'PR', 'EFFECTIVE_STOCK']
        date_cols = ['LAST_ENTRY_DATE', 'First_Inv_Year']

        keep_cols = base_cols + [c for c in stock_cols + date_cols if c in latest.columns]
        sku_base = latest[keep_cols].copy()

        # Calculate EFFECTIVE_STOCK if not present
        if 'EFFECTIVE_STOCK' not in sku_base.columns:
            sku_base['EFFECTIVE_STOCK'] = (
                sku_base.get('CURRENT_STOCK', 0) +
                sku_base.get('OUTSTANDING', 0) +
                sku_base.get('PR', 0)
            )

        # Last sale date (vectorized)
        last_sale = supplier_df.groupby('SKU')['Date'].max().reset_index()
        last_sale.columns = ['SKU', 'Last_Sale_Date']
        sku_base = sku_base.merge(last_sale, on='SKU', how='left')

        print(f"   ✅ Base table: {len(sku_base):,} SKUs")

        # ═══════════════════════════════════════════════════════════════
        # 5. Sales Aggregations (Vectorized)
        # ═══════════════════════════════════════════════════════════════

        print("\n📊 Step 5: Aggregating sales by year...")

        # Total sales per SKU per year
        yearly = supplier_df.groupby(['SKU', 'Year']).agg({
            'bal Qty': 'sum',
            'bal Value': 'sum',
            'Total_Cost': 'sum',
            'Total_Profit': 'sum'
        }).reset_index()

        # Pivot quantities
        qty_pivot = yearly.pivot(
            index='SKU',
            columns='Year',
            values='bal Qty'
        ).fillna(0)

        qty_pivot.columns = [f'Qty_{int(y)}' for y in qty_pivot.columns]
        qty_pivot = qty_pivot.reset_index()

        # Pivot values
        val_pivot = yearly.pivot(
            index='SKU',
            columns='Year',
            values='bal Value'
        ).fillna(0)

        val_pivot.columns = [f'Sales_{int(y)}' for y in val_pivot.columns]
        val_pivot = val_pivot.reset_index()

        # Merge
        sku_base = sku_base.merge(qty_pivot, on='SKU', how='left')
        sku_base = sku_base.merge(val_pivot, on='SKU', how='left')

        print("   ✅ Yearly sales added")

        # ═══════════════════════════════════════════════════════════════
        # 6. 2025 Breakdown by Type (CONTRACT vs FAMILY)
        # ═══════════════════════════════════════════════════════════════

        print("\n🏢 Step 6: Breaking down 2025 by sales type...")

        current_year = datetime.now().year

        sales_2025 = supplier_df[supplier_df['Year'] == current_year].copy()

        if len(sales_2025) > 0 and 'Sales_Type' in sales_2025.columns:
            type_agg = sales_2025.groupby(['SKU', 'Sales_Type']).agg({
                'bal Qty': 'sum',
                'bal Value': 'sum'
            }).reset_index()

            # Pivot by type
            type_qty = type_agg.pivot(
                index='SKU',
                columns='Sales_Type',
                values='bal Qty'
            ).fillna(0).reset_index()

            type_qty.columns = ['SKU'] + [f'Qty_{current_year}_{c}' for c in type_qty.columns[1:]]

            sku_base = sku_base.merge(type_qty, on='SKU', how='left')

            print(f"   ✅ 2025 breakdown added")
        else:
            print("   ⚠️  No 2025 data or Sales_Type column")

        # ═══════════════════════════════════════════════════════════════
        # 7. Merge Forecast Data
        # ═══════════════════════════════════════════════════════════════

        if self.forecast_df is not None:
            print("\n🔮 Step 7: Merging forecast data...")

            forecast_df = self.forecast_df.copy()
            forecast_df['SKU'] = normalize_sku(forecast_df['SKU'])

            # Select forecast columns
            fcst_cols = ['SKU', 'Pattern', 'Forecast_Daily', 'Forecast_30D',
                        'ROP', 'Max_Stock', 'Safety_Stock']

            available_fcst_cols = [c for c in fcst_cols if c in forecast_df.columns]

            sku_base = sku_base.merge(
                forecast_df[available_fcst_cols],
                on='SKU',
                how='left'
            )

            print(f"   ✅ Forecast data merged")

        # ═══════════════════════════════════════════════════════════════
        # 8. Collection-Level Aggregations (Vectorized)
        # ═══════════════════════════════════════════════════════════════

        print("\n📚 Step 8: Aggregating collection metrics...")

        # Collection sales by year
        coll_yearly = supplier_df.groupby(['Collection', 'Year']).agg({
            'bal Value': 'sum',
            'bal Qty': 'sum'
        }).reset_index()

        coll_pivot = coll_yearly.pivot(
            index='Collection',
            columns='Year',
            values='bal Value'
        ).fillna(0).reset_index()

        coll_pivot.columns = ['Collection'] + [
            f'Coll_Sales_{int(y)}' for y in coll_pivot.columns[1:]
        ]

        sku_base = sku_base.merge(coll_pivot, on='Collection', how='left')

        # Collection total SKUs
        coll_sku_count = sku_base.groupby('Collection').size().reset_index()
        coll_sku_count.columns = ['Collection', 'Collection_SKU_Count']

        sku_base = sku_base.merge(coll_sku_count, on='Collection', how='left')

        print("   ✅ Collection metrics added")

        # ═══════════════════════════════════════════════════════════════
        # 9. Calculate Realistic Metrics (Vectorized)
        # ═════════════════════════════════════════════════════════════

        print("\n🎯 Step 9: Calculating realistic metrics...")

        # Days since last sale
        sku_base['Days_Since_Last_Sale'] = (
            pd.Timestamp.now() - sku_base['Last_Sale_Date']
        ).dt.days

        # Collection age
        current_year = datetime.now().year
        sku_base['Collection_Age_Years'] = (
            current_year - sku_base['First_Inv_Year']
        )

        # Stock value
        sku_base['Stock_Value'] = (
            sku_base['CURRENT_STOCK'] * sku_base['Final_Cost']
        )

        # COGS
        sku_base['COGS'] = sku_base['Stock_Value']

        # Stock turnover (realistic calculation)
        # Annual sales / Average stock value
        if f'Sales_{current_year}' in sku_base.columns:
            sku_base['Stock_Turnover_Ratio'] = np.where(
                sku_base['Stock_Value'] > 0,
                sku_base[f'Sales_{current_year}'] / sku_base['Stock_Value'],
                0
            )

        # Stock status classification
        sku_base['Stock_Status'] = 'Normal'

        if 'ROP' in sku_base.columns:
            sku_base.loc[sku_base['EFFECTIVE_STOCK'] < sku_base['ROP'], 'Stock_Status'] = 'Below_ROP'
            sku_base.loc[sku_base['CURRENT_STOCK'] == 0, 'Stock_Status'] = 'Out_of_Stock'

            if 'Max_Stock' in sku_base.columns:
                sku_base.loc[sku_base['EFFECTIVE_STOCK'] > sku_base['Max_Stock'], 'Stock_Status'] = 'Overstock'

        # Recommended order
        if 'Max_Stock' in sku_base.columns:
            sku_base['Recommended_Order'] = np.maximum(
                0,
                sku_base['Max_Stock'] - sku_base['EFFECTIVE_STOCK']
            ).round(0)

        # Activity classification based on recency and frequency
        sku_base['Activity_Level'] = 'Inactive'

        if f'Qty_{current_year}' in sku_base.columns:
            # Active: Sold in current year
            sku_base.loc[sku_base[f'Qty_{current_year}'] > 0, 'Activity_Level'] = 'Active'

            # Very Active: High volume + recent sales
            high_volume = sku_base[f'Qty_{current_year}'] > sku_base[f'Qty_{current_year}'].quantile(0.75)
            recent_sales = sku_base['Days_Since_Last_Sale'] < 30

            sku_base.loc[high_volume & recent_sales, 'Activity_Level'] = 'Very_Active'

            # Slow Moving: Some sales but low volume
            slow = (sku_base[f'Qty_{current_year}'] > 0) & (sku_base[f'Qty_{current_year}'] < sku_base[f'Qty_{current_year}'].quantile(0.25))
            sku_base.loc[slow, 'Activity_Level'] = 'Slow_Moving'

        print("   ✅ Metrics calculated")

        # ═══════════════════════════════════════════════════════════════
        # 10. Clean and Organize
        # ═══════════════════════════════════════════════════════════════

        print("\n📋 Step 10: Organizing output...")

        # Fill NaN
        sku_base = sku_base.fillna(0)

        # Round numeric columns
        numeric_cols = sku_base.select_dtypes(include=[np.number]).columns
        sku_base[numeric_cols] = sku_base[numeric_cols].round(2)

        print(f"   ✅ Final dataset: {len(sku_base):,} SKUs × {len(sku_base.columns)} columns")

        # ═══════════════════════════════════════════════════════════════
        # 11. Save Results
        # ═══════════════════════════════════════════════════════════════

        print("\n💾 Step 11: Saving results...")

        # Clean supplier name for filename
        safe_name = "".join(c for c in supplier_name if c.isalnum() or c in (' ', '-', '_'))
        safe_name = safe_name[:50]

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

        # Save Excel
        excel_file = self.output_folder / f"Supplier_Analysis_{safe_name}_{timestamp}.xlsx"

        with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
            # Main data
            sku_base.to_excel(writer, sheet_name='SKU_Analysis', index=False)

            # Summary sheet
            summary = self._create_summary(sku_base, supplier_name)
            summary.to_excel(writer, sheet_name='Summary', index=False)

            # Action items
            if 'Stock_Status' in sku_base.columns:
                action_items = sku_base[
                    sku_base['Stock_Status'].isin(['Below_ROP', 'Out_of_Stock'])
                ].sort_values('Recommended_Order', ascending=False)

                if len(action_items) > 0:
                    action_items.to_excel(writer, sheet_name='Action_Items', index=False)

        print(f"   ✅ Excel saved: {excel_file}")

        # Generate PDF
        if PDF_OK:
            pdf_file = self._generate_pdf_report(sku_base, supplier_name, safe_name, timestamp)
            if pdf_file:
                print(f"   ✅ PDF saved: {pdf_file}")

        print("\n" + "=" * 80)
        print("✅ ANALYSIS COMPLETE!")
        print("=" * 80)

        return sku_base

    def _create_summary(self, sku_df, supplier_name):
        """Create summary statistics"""

        summary_data = []

        # Basic metrics
        summary_data.append(['Supplier', supplier_name])
        summary_data.append(['Total SKUs', len(sku_df)])
        summary_data.append(['Total Collections', sku_df['Collection'].nunique()])

        # Stock metrics
        summary_data.append(['Total Stock Value', sku_df['Stock_Value'].sum()])
        summary_data.append(['Total Current Stock', sku_df['CURRENT_STOCK'].sum()])
        summary_data.append(['Total Effective Stock', sku_df['EFFECTIVE_STOCK'].sum()])

        # Sales metrics
        current_year = datetime.now().year
        if f'Sales_{current_year}' in sku_df.columns:
            summary_data.append([f'Total Sales {current_year}', sku_df[f'Sales_{current_year}'].sum()])
            summary_data.append([f'Total Qty {current_year}', sku_df[f'Qty_{current_year}'].sum()])

        # Stock status
        if 'Stock_Status' in sku_df.columns:
            summary_data.append(['SKUs Below ROP', (sku_df['Stock_Status'] == 'Below_ROP').sum()])
            summary_data.append(['SKUs Out of Stock', (sku_df['Stock_Status'] == 'Out_of_Stock').sum()])
            summary_data.append(['SKUs Overstock', (sku_df['Stock_Status'] == 'Overstock').sum()])

        # Activity level
        if 'Activity_Level' in sku_df.columns:
            summary_data.append(['Very Active SKUs', (sku_df['Activity_Level'] == 'Very_Active').sum()])
            summary_data.append(['Active SKUs', (sku_df['Activity_Level'] == 'Active').sum()])
            summary_data.append(['Slow Moving SKUs', (sku_df['Activity_Level'] == 'Slow_Moving').sum()])
            summary_data.append(['Inactive SKUs', (sku_df['Activity_Level'] == 'Inactive').sum()])

        return pd.DataFrame(summary_data, columns=['Metric', 'Value'])

    def _generate_pdf_report(self, sku_df, supplier_name, safe_name, timestamp):
        """Generate professional PDF report"""

        pdf_file = self.output_folder / f"Supplier_Report_{safe_name}_{timestamp}.pdf"

        doc = SimpleDocTemplate(
            str(pdf_file),
            pagesize=A4,
            topMargin=0.6*inch,
            bottomMargin=0.6*inch,
            leftMargin=0.5*inch,
            rightMargin=0.5*inch
        )

        story = []
        styles = getSampleStyleSheet()

        # Custom styles
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=24,
            textColor=rl_colors.HexColor('#2c3e50'),
            alignment=TA_CENTER,
            spaceAfter=12
        )

        heading_style = ParagraphStyle(
            'CustomHeading',
            parent=styles['Heading2'],
            fontSize=14,
            textColor=rl_colors.HexColor('#34495e'),
            spaceBefore=12,
            spaceAfter=8
        )

        # Title
        story.append(Paragraph("Supplier Analysis Report", title_style))
        story.append(Paragraph(
            f"{supplier_name}",
            ParagraphStyle('Subtitle', parent=styles['Normal'],
                         fontSize=16, alignment=TA_CENTER, textColor=rl_colors.HexColor('#7f8c8d'))
        ))
        story.append(Paragraph(
            f"Generated: {datetime.now().strftime('%B %d, %Y at %H:%M')}",
            ParagraphStyle('Date', parent=styles['Normal'],
                         fontSize=10, alignment=TA_CENTER, textColor=rl_colors.grey)
        ))
        story.append(Spacer(1, 0.3*inch))

        # Executive Summary
        story.append(Paragraph("Executive Summary", heading_style))

        summary_data = [
            ['Metric', 'Value'],
            ['Total SKUs', f"{len(sku_df):,}"],
            ['Total Collections', f"{sku_df['Collection'].nunique():,}"],
            ['Current Stock Value', f"{sku_df['Stock_Value'].sum():,.0f} SAR"],
            ['Total Units in Stock', f"{sku_df['CURRENT_STOCK'].sum():,.0f}"],
        ]

        # Add current year sales
        current_year = datetime.now().year
        if f'Sales_{current_year}' in sku_df.columns:
            summary_data.append([f'{current_year} Sales', f"{sku_df[f'Sales_{current_year}'].sum():,.0f} SAR"])
            summary_data.append([f'{current_year} Qty Sold', f"{sku_df[f'Qty_{current_year}'].sum():,.0f}"])

        summary_table = Table(summary_data, colWidths=[3*inch, 3*inch])
        summary_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), rl_colors.HexColor('#34495e')),
            ('TEXTCOLOR', (0,0), (-1,0), rl_colors.whitesmoke),
            ('ALIGN', (0,0), (-1,-1), 'LEFT'),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
            ('FONTSIZE', (0,0), (-1,0), 10),
            ('FONTSIZE', (0,1), (-1,-1), 9),
            ('GRID', (0,0), (-1,-1), 0.5, rl_colors.grey),
            ('ROWBACKGROUNDS', (0,1), (-1,-1), [rl_colors.white, rl_colors.lightgrey])
        ]))

        story.append(summary_table)
        story.append(Spacer(1, 0.2*inch))

        # Stock Status
        if 'Stock_Status' in sku_df.columns:
            story.append(Paragraph("Stock Status", heading_style))

            status_counts = sku_df['Stock_Status'].value_counts()
            status_data = [['Status', 'Count', '%']]

            for status, count in status_counts.items():
                pct = (count / len(sku_df) * 100)
                status_data.append([status, f"{count:,}", f"{pct:.1f}%"])

            status_table = Table(status_data, colWidths=[2*inch, 2*inch, 2*inch])
            status_table.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), rl_colors.HexColor('#3498db')),
                ('TEXTCOLOR', (0,0), (-1,0), rl_colors.whitesmoke),
                ('ALIGN', (0,0), (-1,-1), 'CENTER'),
                ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
                ('FONTSIZE', (0,0), (-1,-1), 9),
                ('GRID', (0,0), (-1,-1), 0.5, rl_colors.grey),
                ('ROWBACKGROUNDS', (0,1), (-1,-1), [rl_colors.white, rl_colors.lightgrey])
            ]))

            story.append(status_table)
            story.append(Spacer(1, 0.2*inch))

        # Top 15 SKUs by Sales
        story.append(Paragraph("Top 15 SKUs by 2025 Sales", heading_style))

        if f'Sales_{current_year}' in sku_df.columns:
            top_skus = sku_df.nlargest(15, f'Sales_{current_year}')[
                ['SKU', 'Collection', f'Sales_{current_year}', f'Qty_{current_year}',
                 'CURRENT_STOCK', 'Stock_Status']
            ]

            top_data = [['SKU', 'Collection', 'Sales', 'Qty', 'Stock', 'Status']]

            for _, row in top_skus.iterrows():
                top_data.append([
                    str(row['SKU'])[:15],
                    str(row['Collection']),
                    f"{row[f'Sales_{current_year}']:,.0f}",
                    f"{row[f'Qty_{current_year}']:,.0f}",
                    f"{row['CURRENT_STOCK']:,.0f}",
                    str(row.get('Stock_Status', 'N/A'))
                ])

            top_table = Table(top_data, colWidths=[1.3*inch, 0.9*inch, 1*inch, 0.8*inch, 0.8*inch, 1*inch])
            top_table.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), rl_colors.HexColor('#2ecc71')),
                ('TEXTCOLOR', (0,0), (-1,0), rl_colors.whitesmoke),
                ('ALIGN', (0,0), (-1,-1), 'CENTER'),
                ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
                ('FONTSIZE', (0,0), (-1,-1), 7),
                ('GRID', (0,0), (-1,-1), 0.5, rl_colors.grey),
                ('ROWBACKGROUNDS', (0,1), (-1,-1), [rl_colors.white, rl_colors.lightgrey])
            ]))

            story.append(top_table)

        # Build PDF
        try:
            doc.build(story)
            return pdf_file
        except Exception as e:
            print(f"   ⚠️  PDF generation error: {e}")
            return None


# ════════════════════════════════════════════════════════════════════════════
# SIMPLE WRAPPER FUNCTION
# ════════════════════════════════════════════════════════════════════════════

def analyze_supplier(
    combined_df,
    supplier_name,
    forecast_df=None,
    adjusted_cost_file=None
):
    """
    Simple wrapper for supplier analysis

    Parameters:
    -----------
    combined_df : DataFrame
        Main sales data
    supplier_name : str
        Supplier to analyze
    forecast_df : DataFrame, optional
        Forecast results from ForecastingPipeline
    adjusted_cost_file : str, optional
        Path to adjusted cost Excel file

    Returns:
    --------
    result_df : DataFrame
        Complete supplier analytics

    Example:
    --------
    result = analyze_supplier(
        combined_df=combined_df,
        supplier_name="D'DECOR HOME FABRICS PVT.LTD",
        forecast_df=sku_forecasts,
        adjusted_cost_file='C:/Users/User/Desktop/ADJUSTED_COST.xlsx'
    )
    """

    analyzer = SupplierAnalyzer(
        combined_df=combined_df,
        forecast_df=forecast_df
    )

    result = analyzer.analyze_supplier(
        supplier_name=supplier_name,
        adjusted_cost_file=adjusted_cost_file
    )

    return result


# ════════════════════════════════════════════════════════════════════════════
# USAGE
# ════════════════════════════════════════════════════════════════════════════

print("\n✅ Supplier Analysis System Ready!")
print()
print("🚀 USAGE:")
print()
print("result = analyze_supplier(")
print("    combined_df=combined_df,")
print("    supplier_name=\"D'DECOR HOME FABRICS PVT.LTD\",")
print("    forecast_df=sku_forecasts,  # Optional")
print("    adjusted_cost_file='path/to/adjusted_cost.xlsx'  # Optional")
print(")")
print()
print("📁 Outputs:")
print("   • Excel: Desktop/supplier_analysis/Supplier_Analysis_*.xlsx")
print("   • PDF:   Desktop/supplier_analysis/Supplier_Report_*.pdf")
print()
print("=" * 80)

📊 PROFESSIONAL SUPPLIER ANALYSIS SYSTEM

✅ Supplier Analysis System Ready!

🚀 USAGE:

result = analyze_supplier(
    combined_df=combined_df,
    supplier_name="D'DECOR HOME FABRICS PVT.LTD",
    forecast_df=sku_forecasts,  # Optional
    adjusted_cost_file='path/to/adjusted_cost.xlsx'  # Optional
)

📁 Outputs:
   • Excel: Desktop/supplier_analysis/Supplier_Analysis_*.xlsx
   • PDF:   Desktop/supplier_analysis/Supplier_Report_*.pdf



In [18]:
result_supplier = analyze_supplier(
    combined_df=combined_df,
    supplier_name="D'DECOR HOME FABRICS PVT.LTD",
    forecast_df=inventory_df_tagropa,  # Optional
    adjusted_cost_file=r'C:\Users\User\Desktopadjusted_cost.xlsx'  # Optional
)


📊 ANALYZING SUPPLIER: D'DECOR HOME FABRICS PVT.LTD

🔍 Step 1: Filtering supplier data...
   ✅ Found 723,336 records
   ✅ SKUs: 9,716
   ✅ Period: 2019-01-01 → 2026-01-14

💵 Step 3: Calculating financials...
   ✅ Financials calculated

📦 Step 4: Building SKU-level analytics...
   ✅ Base table: 9,716 SKUs

📊 Step 5: Aggregating sales by year...
   ✅ Yearly sales added

🏢 Step 6: Breaking down 2025 by sales type...
   ✅ 2025 breakdown added

🔮 Step 7: Merging forecast data...
   ✅ Forecast data merged

📚 Step 8: Aggregating collection metrics...
   ✅ Collection metrics added

🎯 Step 9: Calculating realistic metrics...
   ✅ Metrics calculated

📋 Step 10: Organizing output...
   ✅ Final dataset: 9,716 SKUs × 47 columns

💾 Step 11: Saving results...


KeyError: 'Recommended_Order'

In [ ]:
combined_df.columns

## DASHBOARD FOR  ANALYSIS BY SUPPLIER

In [2]:
# ════════════════════════════════════════════════════════════════════════════
# SUPPLIER DASHBOARD - INTERACTIVE ANALYTICS
# Real-time visualization + KPIs + Drill-down capabilities
# ════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("📊 SUPPLIER DASHBOARD SYSTEM")
print("=" * 80)


# ════════════════════════════════════════════════════════════════════════════
# SUPPLIER DASHBOARD CLASS
# ════════════════════════════════════════════════════════════════════════════

class SupplierDashboard:
    """Interactive supplier dashboard with real-time analytics"""

    def __init__(self, combined_df, supplier_name, forecast_df=None):
        print(f"\n🔧 Initializing dashboard for: {supplier_name}")

        self.df = combined_df.copy()
        self.supplier_name = supplier_name
        self.forecast_df = forecast_df

        # Filter supplier data
        self.supplier_df = self.df[self.df['Supplier'] == supplier_name].copy()

        if len(self.supplier_df) == 0:
            print(f"❌ No data for supplier: {supplier_name}")
            return

        # Ensure date column
        self.supplier_df['Date'] = pd.to_datetime(self.supplier_df['Date'])
        self.supplier_df['Year'] = self.supplier_df['Date'].dt.year
        self.supplier_df['Month'] = self.supplier_df['Date'].dt.month
        self.supplier_df['YearMonth'] = self.supplier_df['Date'].dt.to_period('M')

        # Parse Collection
        if 'Collection' not in self.supplier_df.columns:
            self.supplier_df['Section'] = self.supplier_df['SKU'].astype(str).str[:4]

        # Create widgets
        self._create_widgets()

        print(f"✅ Dashboard ready!")
        print(f"   Records: {len(self.supplier_df):,}")
        print(f"   SKUs: {self.supplier_df['SKU'].nunique():,}")
        print(f"   Collections: {self.supplier_df['Section'].nunique():,}")
        print(f"   Period: {self.supplier_df['Date'].min().date()} → {self.supplier_df['Date'].max().date()}")

    def _create_widgets(self):
        """Create interactive widgets"""

        # Year selector
        years = sorted(self.supplier_df['Year'].unique(), reverse=True)
        self.year_selector = widgets.SelectMultiple(
            options=years,
            value=[years[0]] if len(years) > 0 else [],
            description='Years:',
            disabled=False,
            layout=widgets.Layout(width='200px', height='100px')
        )

        # Collection filter
        collections = sorted(self.supplier_df['Section'].unique())
        self.collection_filter = widgets.SelectMultiple(
            options=['ALL'] + collections,
            value=['ALL'],
            description='Collections:',
            disabled=False,
            layout=widgets.Layout(width='200px', height='150px')
        )

        # Metric selector
        self.metric_selector = widgets.Dropdown(
            options=['Sales Value', 'Quantity', 'Profit', 'Margin %'],
            value='Sales Value',
            description='Metric:',
            disabled=False,
            layout=widgets.Layout(width='300px')
        )

        # Update button
        self.update_btn = widgets.Button(
            description='🔄 Update Dashboard',
            button_style='primary',
            layout=widgets.Layout(width='200px', height='40px')
        )
        self.update_btn.on_click(self._update_dashboard)

        # Output area
        self.output = widgets.Output()

    def _update_dashboard(self, b=None):
        """Update dashboard with current filters"""

        with self.output:
            clear_output(wait=True)

            # Get filter values
            selected_years = list(self.year_selector.value)
            selected_collections = list(self.collection_filter.value)

            if not selected_years:
                print("⚠️  Please select at least one year")
                return

            # Filter data
            filtered_df = self.supplier_df[
                self.supplier_df['Year'].isin(selected_years)
            ].copy()

            if 'ALL' not in selected_collections:
                filtered_df = filtered_df[
                    filtered_df['Section'].isin(selected_collections)
                ]

            if len(filtered_df) == 0:
                print("⚠️  No data for selected filters")
                return

            # Display dashboard
            self._display_dashboard(filtered_df, selected_years)

    def _display_dashboard(self, df, selected_years):
        """Display complete dashboard"""

        # Header
        self._show_header()

        # KPIs
        self._show_kpis(df, selected_years)

        # Charts
        self._show_charts(df, selected_years)

        # Tables
        self._show_tables(df)

    def _show_header(self):
        """Display dashboard header"""

        html = f"""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 30px; border-radius: 15px; margin-bottom: 20px;
        box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
            <h1 style='color: #FFF; margin: 0; font-size: 36px; text-align: center;
            font-weight: 900; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);'>
            📊 Supplier Dashboard</h1>
            <p style='color: #FFF; margin: 10px 0 0; font-size: 18px; text-align: center;
            font-weight: 600;'>{self.supplier_name}</p>
            <p style='color: #FFF; margin: 5px 0 0; font-size: 14px; text-align: center;'>
            {datetime.now().strftime('%B %Y')}</p>
        </div>
        """
        display(HTML(html))

    def _show_kpis(self, df, selected_years):
        """Display KPI cards"""

        # Calculate KPIs
        total_sales = df['bal Value'].sum()
        total_qty = df['bal Qty'].sum()

        total_profit = 0
        avg_margin = 0
        if 'Total_Profit' in df.columns:
            total_profit = df['Total_Profit'].sum()
            avg_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0

        unique_skus = df['SKU'].nunique()
        unique_collections = df['Collection'].nunique()

        # Get current stock info
        latest = df.sort_values('Date').groupby('SKU').last()
        total_stock = latest['CURRENT_STOCK'].sum() if 'CURRENT_STOCK' in latest.columns else 0
        total_effective = latest['EFFECTIVE_STOCK'].sum() if 'EFFECTIVE_STOCK' in latest.columns else 0

        # Stock value
        if 'Cost' in latest.columns:
            stock_value = (latest['CURRENT_STOCK'] * latest['Cost']).sum()
        else:
            stock_value = 0

        # HTML
        html = """
        <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
        gap: 15px; margin-bottom: 20px;'>
        """

        cards = [
            ('💰 Total Sales', f'{total_sales:,.0f}', 'SAR', '#667eea', '#764ba2'),
            ('📦 Quantity Sold', f'{total_qty:,.0f}', 'Units', '#f093fb', '#f5576c'),
            ('💎 Total Profit', f'{total_profit:,.0f}', f'{avg_margin:.1f}%', '#fa709a', '#fee140'),
            ('📊 Active SKUs', f'{unique_skus:,}', 'Items', '#43e97b', '#38f9d7'),
            ('📚 Collections', f'{unique_collections:,}', 'Groups', '#4facfe', '#00f2fe'),
            ('🏪 Current Stock', f'{total_stock:,.0f}', 'Units', '#a8edea', '#fed6e3'),
            ('✅ Effective Stock', f'{total_effective:,.0f}', 'Total', '#30cfd0', '#330867'),
            ('💵 Stock Value', f'{stock_value:,.0f}', 'SAR', '#ff6e7f', '#bfe9ff')
        ]

        for title, value, subtitle, c1, c2 in cards:
            html += f"""
            <div style='background: linear-gradient(135deg, {c1} 0%, {c2} 100%);
            padding: 20px; border-radius: 10px; color: white;
            box-shadow: 0 5px 15px rgba(0,0,0,0.2);'>
                <div style='font-size: 12px; font-weight: 600; margin-bottom: 8px;'>{title}</div>
                <div style='font-size: 28px; font-weight: 900;
                text-shadow: 2px 2px 4px rgba(0,0,0,0.3);'>{value}</div>
                <div style='font-size: 11px; font-weight: 600; margin-top: 5px;'>{subtitle}</div>
            </div>
            """

        html += "</div>"
        display(HTML(html))

    def _show_charts(self, df, selected_years):
        """Display interactive charts"""

        # 1. Monthly Trend
        monthly = df.groupby('YearMonth').agg({
            'bal Value': 'sum',
            'bal Qty': 'sum',
            'Total_Profit': 'sum' if 'Total_Profit' in df.columns else lambda x: 0
        }).reset_index()

        monthly['YearMonth'] = monthly['YearMonth'].astype(str)

        fig1 = make_subplots(
            rows=1, cols=2,
            subplot_titles=('Monthly Sales Trend', 'Monthly Quantity Trend')
        )

        fig1.add_trace(
            go.Scatter(
                x=monthly['YearMonth'],
                y=monthly['bal Value'],
                mode='lines+markers',
                name='Sales',
                line=dict(color='#667eea', width=3),
                marker=dict(size=8)
            ),
            row=1, col=1
        )

        fig1.add_trace(
            go.Scatter(
                x=monthly['YearMonth'],
                y=monthly['bal Qty'],
                mode='lines+markers',
                name='Quantity',
                line=dict(color='#f093fb', width=3),
                marker=dict(size=8)
            ),
            row=1, col=2
        )

        fig1.update_layout(
            height=400,
            showlegend=False,
            template='plotly_white',
            title_text='Sales Performance Over Time'
        )

        display(fig1)

        # 2. Top Collections
        coll_sales = df.groupby('Collection').agg({
            'bal Value': 'sum',
            'bal Qty': 'sum'
        }).reset_index()

        coll_sales = coll_sales.nlargest(15, 'bal Value')

        fig2 = go.Figure()

        fig2.add_trace(go.Bar(
            x=coll_sales['Collection'],
            y=coll_sales['bal Value'],
            marker_color='#43e97b',
            text=coll_sales['bal Value'].apply(lambda x: f'{x/1000:.0f}K'),
            textposition='outside'
        ))

        fig2.update_layout(
            title='Top 15 Collections by Sales',
            xaxis_title='Collection',
            yaxis_title='Sales (SAR)',
            height=400,
            template='plotly_white'
        )

        display(fig2)

        # 3. Sales by Type (if available)
        if 'Sales_Type' in df.columns:
            type_sales = df.groupby('Sales_Type').agg({
                'bal Value': 'sum',
                'bal Qty': 'sum'
            }).reset_index()

            fig3 = make_subplots(
                rows=1, cols=2,
                specs=[[{'type':'pie'}, {'type':'pie'}]],
                subplot_titles=('Sales Value by Type', 'Quantity by Type')
            )

            fig3.add_trace(
                go.Pie(
                    labels=type_sales['Sales_Type'],
                    values=type_sales['bal Value'],
                    marker_colors=['#667eea', '#f093fb']
                ),
                row=1, col=1
            )

            fig3.add_trace(
                go.Pie(
                    labels=type_sales['Sales_Type'],
                    values=type_sales['bal Qty'],
                    marker_colors=['#43e97b', '#fa709a']
                ),
                row=1, col=2
            )

            fig3.update_layout(
                height=350,
                template='plotly_white',
                title_text='CONTRACT vs FAMILY Distribution'
            )

            display(fig3)

        # 4. Year-over-Year Comparison
        if len(selected_years) > 1:
            yearly = df.groupby('Year').agg({
                'bal Value': 'sum',
                'bal Qty': 'sum'
            }).reset_index()

            fig4 = go.Figure()

            fig4.add_trace(go.Bar(
                x=yearly['Year'],
                y=yearly['bal Value'],
                name='Sales',
                marker_color='#4facfe',
                text=yearly['bal Value'].apply(lambda x: f'{x/1000:.0f}K'),
                textposition='outside'
            ))

            fig4.update_layout(
                title='Year-over-Year Sales Comparison',
                xaxis_title='Year',
                yaxis_title='Sales (SAR)',
                height=350,
                template='plotly_white'
            )

            display(fig4)

    def _show_tables(self, df):
        """Display summary tables"""

        # Top 20 SKUs
        html = """
        <div style='background: white; padding: 20px; border-radius: 10px;
        margin-top: 20px; box-shadow: 0 4px 12px rgba(0,0,0,0.1);'>
            <h2 style='margin-top: 0; color: #2c3e50; font-weight: 900;'>
            📦 Top 20 SKUs by Sales</h2>
        """

        top_skus = df.groupby('SKU').agg({
            'bal Value': 'sum',
            'bal Qty': 'sum',
            'Collection': 'first'
        }).reset_index()

        top_skus = top_skus.nlargest(20, 'bal Value')

        # Get latest stock
        latest = df.sort_values('Date').groupby('SKU').last()

        html += """
        <table style='width: 100%; border-collapse: collapse; font-size: 12px;'>
            <thead>
                <tr style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;'>
                    <th style='padding: 10px; text-align: left;'>#</th>
                    <th style='padding: 10px; text-align: left;'>SKU</th>
                    <th style='padding: 10px; text-align: left;'>Collection</th>
                    <th style='padding: 10px; text-align: right;'>Sales</th>
                    <th style='padding: 10px; text-align: right;'>Qty</th>
                    <th style='padding: 10px; text-align: right;'>Stock</th>
                </tr>
            </thead>
            <tbody>
        """

        for i, (_, row) in enumerate(top_skus.iterrows(), 1):
            bg = '#f8f9fa' if i % 2 == 0 else 'white'

            stock = latest.loc[row['SKU'], 'CURRENT_STOCK'] if row['SKU'] in latest.index and 'CURRENT_STOCK' in latest.columns else 0

            html += f"""
            <tr style='background: {bg};'>
                <td style='padding: 8px; font-weight: 700;'>{i}</td>
                <td style='padding: 8px; font-family: monospace;'>{row['SKU']}</td>
                <td style='padding: 8px;'>{row['Collection']}</td>
                <td style='padding: 8px; text-align: right; font-weight: 700;
                color: #00aa00;'>{row['bal Value']:,.0f}</td>
                <td style='padding: 8px; text-align: right;'>{row['bal Qty']:,.0f}</td>
                <td style='padding: 8px; text-align: right;'>{stock:,.0f}</td>
            </tr>
            """

        html += """
            </tbody>
        </table>
        </div>
        """

        display(HTML(html))

    def display(self):
        """Display the dashboard"""

        header = """
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 35px; border-radius: 15px; margin-bottom: 20px;
        box-shadow: 0 12px 35px rgba(0,0,0,0.3); text-align: center;'>
            <h1 style='color: #FFF; margin: 0; font-size: 40px; font-weight: 900;
            text-shadow: 3px 3px 6px rgba(0,0,0,0.4);'>📊 Supplier Dashboard</h1>
            <p style='color: #FFF; margin: 15px 0 0; font-size: 18px; font-weight: 700;'>
            Interactive Analytics & Insights</p>
        </div>
        """

        display(HTML(header))

        # Controls
        controls = widgets.VBox([
            widgets.HTML("<h3 style='margin-top: 0; color: #2c3e50;'>🎛️ Dashboard Controls</h3>"),
            widgets.HBox([self.year_selector, self.collection_filter]),
            self.metric_selector,
            self.update_btn
        ], layout=widgets.Layout(
            padding='20px',
            background_color='#f8f9fa',
            border_radius='10px',
            border='2px solid #e0e0e0'
        ))

        display(controls)
        display(self.output)

        # Auto-display on load
        self._update_dashboard()


# ════════════════════════════════════════════════════════════════════════════
# SIMPLE WRAPPER
# ════════════════════════════════════════════════════════════════════════════

def create_supplier_dashboard(combined_df, supplier_name, forecast_df=None):
    """
    Create interactive supplier dashboard

    Parameters:
    -----------
    combined_df : DataFrame
        Main sales data
    supplier_name : str
        Supplier name
    forecast_df : DataFrame, optional
        Forecast results

    Returns:
    --------
    dashboard : SupplierDashboard
        Dashboard instance

    Example:
    --------
    dashboard = create_supplier_dashboard(
        combined_df=combined_df,
        supplier_name="D'DECOR HOME FABRICS PVT.LTD",
        forecast_df=sku_forecasts
    )

    dashboard.display()
    """

    dashboard = SupplierDashboard(
        combined_df=combined_df,
        supplier_name=supplier_name,
        forecast_df=forecast_df
    )

    return dashboard


# ════════════════════════════════════════════════════════════════════════════
# USAGE
# ════════════════════════════════════════════════════════════════════════════

print("\n✅ Supplier Dashboard System Ready!")
print()
print("🚀 USAGE:")
print()
print("dashboard = create_supplier_dashboard(")
print("    combined_df=combined_df,")
print("    supplier_name=\"D'DECOR HOME FABRICS PVT.LTD\",")
print("    forecast_df=sku_forecasts  # Optional")
print(")")
print()
print("dashboard.display()")
print()
print("=" * 80)

📊 SUPPLIER DASHBOARD SYSTEM

✅ Supplier Dashboard System Ready!

🚀 USAGE:

dashboard = create_supplier_dashboard(
    combined_df=combined_df,
    supplier_name="D'DECOR HOME FABRICS PVT.LTD",
    forecast_df=sku_forecasts  # Optional
)

dashboard.display()



In [11]:
dashboard_supplier = create_supplier_dashboard(
    combined_df=combined_df,
    supplier_name="D'DECOR HOME FABRICS PVT.LTD",
    forecast_df=inventory_df_tagropa  # Optional
)


🔧 Initializing dashboard for: D'DECOR HOME FABRICS PVT.LTD
✅ Dashboard ready!
   Records: 723,336
   SKUs: 9,716
   Collections: 251
   Period: 2019-01-01 → 2026-01-14


In [12]:
dashboard_supplier.display()

Output()